In [1]:
import pandas as pd
import os

RAW   = "raw_data"
CLEAN = "clean_data"
os.makedirs(CLEAN, exist_ok=True)

In [2]:
# ── Load domain mapping from skills.csv ──────────────────────
# skills.csv has two columns: skill_abr, skill_name
# e.g. LGL → Legal, IT → Information Technology
df_map = pd.read_csv(f"{RAW}/skills.csv")
domain_map = dict(zip(df_map['skill_abr'].str.strip(),df_map['skill_name'].str.strip()))
print(f"Loaded {len(domain_map)} domain mappings from skills.csv")

Loaded 35 domain mappings from skills.csv


In [3]:
# ── 1. job_postings ──────────────────────────────────────────
print("\nCleaning job_postings.csv...")
df = pd.read_csv(f"{RAW}/job_postings.csv", low_memory=False)

cols = ['job_id','company_name','title','location','company_id',
        'views','formatted_work_type','applies','remote_allowed',
        'formatted_experience_level','listed_time','work_type',
        'normalized_salary']
df = df[[c for c in cols if c in df.columns]]
df = df.dropna(subset=['job_id','title'])
df = df[df['title'].str.strip() != '']
df['listed_time'] = pd.to_numeric(df['listed_time'], errors='coerce').fillna(0).astype('int64')
for col in ['views','applies']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
df['normalized_salary'] = pd.to_numeric(df['normalized_salary'], errors='coerce')
df = df.replace('', pd.NA).head(5000)
df.to_csv(f"{CLEAN}/job_postings_clean.csv", index=False)
print(f"  Done — {len(df)} rows, {len(df.columns)} columns")


Cleaning job_postings.csv...
  Done — 5000 rows, 13 columns


In [4]:
# ── 2. job_skills ─────────────────────────────────────────────
print("\nCleaning job_skills.csv...")
df = pd.read_csv(f"{RAW}/job_skills.csv")
df = df[['job_id','skill_abr']].dropna()
df = df[df['skill_abr'].str.strip() != '']

# map abbreviations to full names using skills.csv
df['skill_abr'] = df['skill_abr'].str.strip().map(domain_map)
df = df.dropna(subset=['skill_abr'])
df = df.rename(columns={'skill_abr': 'domain'})
df = df.drop_duplicates().head(15000)
df.to_csv(f"{CLEAN}/job_skills_clean.csv", index=False)
print(f"  Done — {len(df)} rows, {df['domain'].nunique()} unique domains")
print(f"  Domains: {sorted(df['domain'].unique())}")


Cleaning job_skills.csv...
  Done — 15000 rows, 35 unique domains
  Domains: ['Accounting/Auditing', 'Administrative', 'Advertising', 'Analyst', 'Art/Creative', 'Business Development', 'Consulting', 'Customer Service', 'Design', 'Distribution', 'Education', 'Engineering', 'Finance', 'General Business', 'Health Care Provider', 'Human Resources', 'Information Technology', 'Legal', 'Management', 'Manufacturing', 'Marketing', 'Other', 'Product Management', 'Production', 'Project Management', 'Public Relations', 'Purchasing', 'Quality Assurance', 'Research', 'Sales', 'Science', 'Strategy/Planning', 'Supply Chain', 'Training', 'Writing/Editing']


In [5]:
# ── 3. companies ──────────────────────────────────────────────
print("\nCleaning companies.csv...")
df = pd.read_csv(f"{RAW}/companies.csv", low_memory=False)
df = df[['company_id','name','company_size','city','country']]
df = df.dropna(subset=['company_id','name'])
df = df.drop_duplicates(subset=['company_id']).head(3000)
df.to_csv(f"{CLEAN}/companies_clean.csv", index=False)
print(f"  Done — {len(df)} rows")


Cleaning companies.csv...
  Done — 3000 rows


In [6]:
# ── 4. salaries ───────────────────────────────────────────────
print("\nCleaning salaries.csv...")
df = pd.read_csv(f"{RAW}/salaries.csv")
df = df.dropna(subset=['job_id'])
for col in ['max_salary','med_salary','min_salary']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df[df['med_salary'].isna() | (df['med_salary'] > 0)]
df = df.drop_duplicates(subset=['job_id']).head(4000)
df.to_csv(f"{CLEAN}/salaries_clean.csv", index=False)
print(f"  Done — {len(df)} rows")


Cleaning salaries.csv...
  Done — 4000 rows


In [7]:
# ── 5. job_industries ─────────────────────────────────────────
print("\nCleaning job_industries.csv...")
df = pd.read_csv(f"{RAW}/job_industries.csv")
df = df[['job_id','industry_id']].dropna().drop_duplicates().head(5000)
df.to_csv(f"{CLEAN}/job_industries_clean.csv", index=False)
print(f"  Done — {len(df)} rows")

print("\nAll 5 files cleaned and saved to /clean_data/")


Cleaning job_industries.csv...
  Done — 5000 rows

All 5 files cleaned and saved to /clean_data/
